In [ ]:
import ee
import geemap
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from geopy.geocoders import Nominatim
import logging
from datetime import date, datetime, timedelta
import ipywidgets as widgets
from IPython.display import display, clear_output

# SYSTEM SETUP & LOGGING
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s', datefmt='%H:%M:%S')
logger = logging.getLogger(__name__)

try:
    ee.Initialize(project='Project_ID')      #replace 'PROJECT_ID' with your own google cloud project id
    logger.info("Earth Engine Initialized.")
except:
    ee.Authenticate()
    ee.Initialize(project='Project_ID')

# INTERACTIVE DEMONSTRATION GUI
Map = geemap.Map(center=[20.0, 0.0], zoom=3)
Map.add_basemap('SATELLITE')

title = widgets.HTML("<h2>VRN Precision Agriculture Simulator (K-Means AI)</h2><p>1. Teleport to location.<br>2. Draw a polygon (from the tools in left of the map).<br>3. Adjust Doses & Scan Days.<br>4. Execute.</p>")

search_box = widgets.Text(placeholder='Village, City, or ZIP code', description='Find Field:', style={'description_width': 'initial'}, layout=widgets.Layout(width='300px'))
search_btn = widgets.Button(description='Teleport', button_style='info', layout=widgets.Layout(width='100px'))
search_ui = widgets.HBox([search_box, search_btn])

time_window_slider = widgets.IntSlider(value=30, min=10, max=120, step=5, description='Scan Window (Days):', style={'description_width': 'initial'})
start_date_picker = widgets.DatePicker(description='Target Date', value=date.today())
dose_low_slider = widgets.FloatSlider(value=100.0, min=0, max=200, step=5, description='Deficient (kg/ha):', style={'description_width': 'initial'})
dose_med_slider = widgets.FloatSlider(value=60.0, min=0, max=200, step=5, description='Standard (kg/ha):', style={'description_width': 'initial'})
blanket_slider = widgets.FloatSlider(value=80.0, min=0, max=200, step=5, description='Blanket (kg/ha):', style={'description_width': 'initial'})
execute_btn = widgets.Button(description='Generate AI Prescription', button_style='success', layout=widgets.Layout(width='100%', height='50px'))
output_log = widgets.Output()

ui_controls = widgets.VBox([title, search_ui, start_date_picker, time_window_slider, dose_low_slider, dose_med_slider, blanket_slider, execute_btn])
display(widgets.HBox([Map, ui_controls]))
display(output_log)

def teleport_to_location(b):
    if search_box.value:
        try:
            geolocator = Nominatim(user_agent="vrn_ag_app", timeout=10)
            location = geolocator.geocode(search_box.value)
            if location:
                Map.set_center(location.longitude, location.latitude, 16)
                with output_log:
                    clear_output(wait=True)
                    print(f"Teleported to: {location.address}")
            else:
                with output_log:
                    print("Location not found.")
        except Exception as e:
            with output_log:
                print(f"Teleport failed: Please check your internet connection or try a broader search term... pin-code works all the time usually. [Error: {e}]")

search_btn.on_click(teleport_to_location)

def on_execute_clicked(b):
    with output_log:
        clear_output(wait=True)
        print(f"Initializing AI Pipeline (Deep-Scanning past {time_window_slider.value} days)...")

        if not Map.draw_features:
            print("Error: How is this supposed to work if you don't specify area. Please draw a shape on the map using the tools on the left.")
            return

        feature = Map.draw_features[-1]
        roi = feature.geometry()

        try:
            # TIME WINDOW SETUP
            picked_date = start_date_picker.value
            if isinstance(picked_date, datetime):
                current_end = picked_date
            elif isinstance(picked_date, date):
                current_end = datetime.combine(picked_date, datetime.min.time())
            else:
                current_end = datetime.strptime(str(picked_date), '%Y-%m-%d')

            start_str = (current_end - timedelta(days=time_window_slider.value)).strftime('%Y-%m-%d')
            end_str = current_end.strftime('%Y-%m-%d')

            # EARTH ENGINE SATELLITE PIPELINE
            def mask_s2_clouds(image):
                qa = image.select('QA60')
                cloudBitMask = 1 << 10
                cirrusBitMask = 1 << 11
                mask = qa.bitwiseAnd(cloudBitMask).eq(0).And(qa.bitwiseAnd(cirrusBitMask).eq(0))
                return image.updateMask(mask)

            def add_ndre(image):
                ndre = image.expression(
                    '(NIR - RED_EDGE) / (NIR + RED_EDGE)',
                    {'NIR': image.select('B8'), 'RED_EDGE': image.select('B5')}
                ).rename('NDRE')
                return image.addBands(ndre)

            s2_collection = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
                             .filterBounds(roi)
                             .filterDate(start_str, end_str)
                             .map(mask_s2_clouds)
                             .map(add_ndre))

            composite = s2_collection.qualityMosaic('NDRE').clip(roi)

            worldcover = ee.ImageCollection("ESA/WorldCover/v200").first()
            cropland_mask = worldcover.eq(40).clip(roi)
            final_ndre = composite.select('NDRE').updateMask(cropland_mask)

            # BASELINE STATS & CV CALCULATION
            stats = final_ndre.reduceRegion(
                reducer=ee.Reducer.mean().combine(reducer2=ee.Reducer.stdDev(), sharedInputs=True),
                geometry=roi,
                scale=10,
                maxPixels=1e9
            ).getInfo()

            if not stats or 'NDRE_mean' not in stats or stats['NDRE_mean'] is None:
                print("Error: No clear crop data found. The field is either fully cloudy or contains no registered crops. Maybe try increasing the Scan Window.")
                return

            mean_val = stats['NDRE_mean']
            std_val = stats['NDRE_stdDev']
            cv = (std_val / mean_val) * 100 if mean_val > 0 else 0

            # SERVER-SIDE MACHINE LEARNING
            # (K-MEANS)
            if cv < 5.0:
                print("Field is highly uniform. Applying blanket nominal zone.")
                zone_img = ee.Image(1).clip(roi).updateMask(final_ndre.mask()).rename('zone')
            else:
                print("Variable field detected. Training K-Means Unsupervised ML model...")

                training_data = final_ndre.sample(region=roi, scale=10, numPixels=1500, tileScale=2)
                clusterer = ee.Clusterer.wekaKMeans(3).train(training_data)
                raw_clusters = final_ndre.cluster(clusterer)

                combined = final_ndre.addBands(raw_clusters)
                cluster_means = combined.reduceRegion(
                    reducer=ee.Reducer.mean().group(groupField=1, groupName='cluster'),
                    geometry=roi, scale=10, maxPixels=1e9
                ).getInfo()

                groups = cluster_means['groups']
                groups_sorted = sorted(groups, key=lambda k: k['mean'])

                # wrapping in ee.List() to increase compatibility on google's servers
                from_list = ee.List([int(g['cluster']) for g in groups_sorted])
                to_list = ee.List([0, 1, 2])

                zone_img = raw_clusters.remap(from_list, to_list).clip(roi).updateMask(final_ndre.mask()).rename('zone')
                print("ML Clustering complete. Natural breaks identified... successfully")

            # AREA & MASS BALANCE
            area_img = ee.Image.pixelArea().addBands(zone_img)
            area_stats = area_img.reduceRegion(
                reducer=ee.Reducer.sum().group(groupField=1, groupName='zone'),
                geometry=roi, scale=10, maxPixels=1e9
            ).getInfo()

            zone_areas_ha = {0: 0.0, 1: 0.0, 2: 0.0}
            if 'groups' in area_stats:
                for g in area_stats['groups']:
                    zone_areas_ha[g['zone']] = g['sum'] / 10000

            ha_low, ha_med, ha_high = zone_areas_ha[0], zone_areas_ha[1], zone_areas_ha[2]
            analyzed_area = ha_low + ha_med + ha_high

            total_vrn = (ha_low * dose_low_slider.value) + (ha_med * dose_med_slider.value)
            total_blanket = analyzed_area * blanket_slider.value
            saved = total_blanket - total_vrn

            # OVERLAY ON INTERACTIVE MAP
            zone_vis = {'min': 0, 'max': 2, 'palette': ['#d7191c', '#ffffbf', '#1a9641']}
            Map.addLayer(zone_img, zone_vis, 'AI Management Zones', True, 0.6)

            # MATPLOTLIB EXTRACTION
            print("Extracting high-resolution arrays for static plotting...")

            rgb_ee = composite.select(['B4', 'B3', 'B2']).divide(3000).clamp(0, 1).unmask(0)
            zone_img_safe = zone_img.unmask(-1)

            rgb_array = geemap.ee_to_numpy(rgb_ee, region=roi, scale=10)
            zone_array = geemap.ee_to_numpy(zone_img_safe, region=roi, scale=10)

            # Report
            print("\n" + "-"*67)
            print(" VARIABLE-RATE NITROGEN (VRN) PRESCRIPTION REPORT")
            print("-"*67)
            print(" SITE CHARACTERIZATION & AI DIAGNOSTICS")
            print(f"  Temporal Scan Window: {time_window_slider.value} Days ({start_str} to {end_str})")
            print(f"  Field Mean NDRE  : {mean_val:.4f}")
            print(f"  Analyzed Crops   : {analyzed_area:.2f} ha (Excluding houses/roads/trees)")
            print(f"  Variance (CV)    : {cv:.1f}%")
            print(f"  Algorithm used   : {'Weka K-Means Unsupervised Clustering' if cv >= 5.0 else 'Uniformity Override'}")
            print("-" * 67)
            print(" MANAGEMENT ZONE DISTRIBUTION")
            print(f"  High Vigor (Green)  : {ha_high:.2f} ha | Dose: 0.00 kg/ha")
            print(f"  Medium Vigor(Yellow): {ha_med:.2f} ha | Dose: {dose_med_slider.value:.2f} kg/ha")
            print(f"  Low Vigor (Red)     : {ha_low:.2f} ha | Dose: {dose_low_slider.value:.2f} kg/ha")
            print("-" * 67)
            print(" AGRONOMIC MASS BALANCE (UREA)")
            print(f"  Conventional Blanket   : {total_blanket:.2f} kg")
            print(f"  Precision VRN Strategy : {total_vrn:.2f} kg")

            if total_blanket > 0:
                print(f"  Material Optimization  : {saved:.2f} kg ({(saved / total_blanket) * 100:.1f}% reduction)")
            print("-"*67 + "\n")

            # STATIC PLOTS
            if rgb_array is not None and zone_array is not None:
                zone_array_2d = np.squeeze(zone_array)

                rx_array = np.full(zone_array_2d.shape, np.nan, dtype=float)
                rx_array[zone_array_2d == 2] = 0.0
                rx_array[zone_array_2d == 1] = dose_med_slider.value
                rx_array[zone_array_2d == 0] = dose_low_slider.value

                plot_zone = np.where(zone_array_2d == -1, np.nan, zone_array_2d)

                fig, axes = plt.subplots(1, 2, figsize=(14, 6))
                cmap_zones = ListedColormap(['#d7191c', '#ffffbf', '#1a9641'])

                axes[0].imshow(np.nan_to_num(rgb_array))
                im1 = axes[0].imshow(plot_zone, cmap=cmap_zones, vmin=0, vmax=2, alpha=0.6, interpolation='none')
                axes[0].set_title('Management Zones Overlay\n(Red=Deficient, Yellow=Nominal, Green=Saturated)')
                axes[0].axis('off')

                axes[1].imshow(np.nan_to_num(rgb_array))
                im2 = axes[1].imshow(rx_array, cmap='YlOrRd', alpha=0.6, interpolation='none')
                axes[1].set_title('Variable-Rate Urea Prescription\n(Overlay)')
                axes[1].axis('off')

                fig.colorbar(im2, ax=axes[1], orientation='vertical', label='Dose Applied (kg / hectare)')

                plt.tight_layout()
                plt.show()

        except Exception as e:
            print(f"\n[SYSTEM EXCEPTION] {e}")

execute_btn.on_click(on_execute_clicked)